# nb_mdf_90_run_all - notebook-only orchestration (no pipeline needed)

Runs Landing -> Bronze -> Silver -> Gold as a dependency graph with `notebookutils.notebook.runMultiple`, sharing
one Spark session (cheap and fast for a demo or for many small tables).

Use this when you want the demo to work in any workspace with zero connections to configure. Use the
**pl_mdf_orchestrator** pipeline to show enterprise orchestration (ForEach, database Copy with watermark, retries, alerts).

Objects are isolated: one failing object does not stop the others (`fail_on_error = False` inside the graph);
this notebook raises at the end if anything failed, so a schedule or pipeline still sees the failure.

In [ ]:
load_group = "all"
run_gold = True

In [ ]:
import json, uuid
from datetime import datetime
orchestration_id = f"orch_{datetime.now():%Y%m%d_%H%M%S}_{uuid.uuid4().hex[:4]}"
common = {"parent_run_id": orchestration_id, "fail_on_error": False}

dag = {
    "activities": [
        {"name": "bronze", "path": "nb_mdf_10_landing_to_bronze", "timeoutPerCellInSeconds": 3600,
         "args": {**common, "load_group": load_group}},
        {"name": "silver", "path": "nb_mdf_20_bronze_to_silver", "timeoutPerCellInSeconds": 3600,
         "args": {**common, "load_group": load_group}, "dependencies": ["bronze"]},
    ],
    "timeoutInSeconds": 7200,
    "concurrency": 2,
}
if run_gold:
    dag["activities"].append({"name": "gold", "path": "nb_mdf_30_silver_to_gold", "timeoutPerCellInSeconds": 3600,
                              "args": {**common}, "dependencies": ["silver"]})

result = notebookutils.notebook.runMultiple(dag, {"displayDAGViaGraphviz": True})
print(json.dumps({k: (v.get("exitVal") if isinstance(v, dict) else str(v)) for k, v in result.items()}, indent=1, default=str))

In [ ]:
summary = spark.sql(f"""SELECT layer, status, COUNT(*) AS objects, SUM(rows_read) AS rows_read, SUM(rows_written) AS rows_written,
                        SUM(rows_quarantined) AS rows_quarantined FROM meta.run_log WHERE parent_run_id = '{orchestration_id}'
                        GROUP BY layer, status ORDER BY layer, status""")
display(summary)
failed = spark.sql(f"SELECT object_id, layer, message FROM meta.run_log WHERE parent_run_id = '{orchestration_id}' AND status = 'FAILED'")
if failed.count():
    display(failed)
    raise Exception(f"{failed.count()} object/layer run(s) failed in {orchestration_id} - see nb_mdf_40_operations_monitor")